# Layer-Swap Ablation — Detectability != Correctability

Highest-priority next experiment for the FACR/LAPSH project.

Decouples **which layer's probe picks boundary-region cases** from **which
layer's CAA vector corrects them**. If detectability != correctability is a
real, general phenomenon (not a Llama+TriviaQA coincidence), you should see:

- accuracy roughly **flat** across detect-layer choice
- accuracy **peaked** at a consistent relative depth across steer-layer choice
- that same flat/peaked pattern holding across Llama-3.2-3B, Qwen-2.5-3B, and Mistral-3B

Set `MODEL_KEY` in the config cell below and run all cells top to bottom.
Change `MODEL_KEY` and re-run for each of the three models — hidden states
and generations are cached to disk so re-running the notebook for the next
model does not redo work you've already done for a previous one, and if a
cell crashes partway you can just re-run from where it stopped.

In [ ]:
!pip install -q transformers datasets scikit-learn pandas tqdm accelerate

## 1. Config

Model configs for Llama / Qwen / Mistral. **Pick which model this run is for** with `MODEL_KEY`.

In [ ]:
import torch

MODEL_CONFIGS = {
    "llama3.2-3b": {
        "hf_name": "meta-llama/Llama-3.2-3B-Instruct",
        "n_layers": 28,
        "hidden_size": 3072,
        "known_detect_layer": 13,   # from prior probe AUROC sweep (~46% depth)
        "known_steer_layer": 24,    # from prior CAA ablation (~86% depth)
    },
    "qwen2.5-3b": {
        "hf_name": "Qwen/Qwen2.5-3B-Instruct",
        "n_layers": 36,
        "hidden_size": 2048,
        "known_detect_layer": None,  # fill in after this notebook's probe sweep
        "known_steer_layer": None,
    },
    "mistral-3b": {
        # NOTE: there's no official Mistral-3B -- verify this repo id before
        # running, swap in whichever community checkpoint you've been using
        "hf_name": "ministral/Ministral-3B-Instruct",
        "n_layers": 32,
        "hidden_size": 3072,
        "known_detect_layer": None,
        "known_steer_layer": None,
    },
}

# ==== SET THIS ====
MODEL_KEY = "llama3.2-3b"   # change to "qwen2.5-3b" or "mistral-3b" and re-run
# ===================

DATASET_NAME = "trivia_qa"
DATASET_SUBSET = "rc.nocontext"
N_TRAIN = 800          # samples used to build probes + CAA vectors
N_EVAL = 200           # held-out samples for ablation accuracy
TOKEN_F1_THRESHOLD = 0.5   # >= this => "factual" label
MAX_NEW_TOKENS = 32
SEED = 42

# alpha (steering strength) sweep -- tuned for Llama's CAA vector norms;
# Qwen/Mistral may need different scaling (watch for logit collapse like you
# saw before with TruthfulQA vectors being much stronger than TriviaQA ones)
ALPHA_GRID = [2.0, 4.0, 6.0, 8.0]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

CACHE_DIR = "./cache"
RESULTS_DIR = "./results"

import os
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

CFG = MODEL_CONFIGS[MODEL_KEY]
HF_NAME = CFG["hf_name"]
N_LAYERS = CFG["n_layers"]

print(f"Running for: {MODEL_KEY}  ({HF_NAME})  n_layers={N_LAYERS}  device={DEVICE}")

## 2. Data loading, generation, and token-F1 factuality labeling

Same label chain you validated before:

```
gold answer -> token_f1(generated, gold) -> factuality_score
factuality_score >= THRESHOLD -> label 1 (factual)
else                          -> label 0 (hallucinated)
```

The probe never sees text, only the hidden state + this binary label.

In [ ]:
import re
import string
import random
import pickle

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(SEED)
torch.manual_seed(SEED)


def normalize_text(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s


def token_f1(pred: str, gold: str) -> float:
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)
    gold_counts = {}
    for t in gold_tokens:
        gold_counts[t] = gold_counts.get(t, 0) + 1
    common = {}
    for t in pred_tokens:
        common[t] = common.get(t, 0) + 1
    num_same = sum(min(c, common.get(t, 0)) for t, c in gold_counts.items())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def best_f1_over_aliases(pred: str, gold_aliases) -> float:
    """TriviaQA gives multiple acceptable aliases; take the max F1."""
    return max(token_f1(pred, g) for g in gold_aliases) if gold_aliases else 0.0


def load_model(hf_name: str):
    tok = AutoTokenizer.from_pretrained(hf_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        hf_name, torch_dtype=DTYPE, output_hidden_states=True
    ).to(DEVICE)
    model.eval()
    return tok, model

In [ ]:
def generate_and_label(model_key: str, hf_name: str, n_total: int):
    cache_path = os.path.join(CACHE_DIR, f"{model_key}_labeled.pkl")
    if os.path.exists(cache_path):
        print(f"[cache hit] loading labeled data from {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    print(f"[generating] {model_key} on {n_total} TriviaQA questions...")
    ds = load_dataset(DATASET_NAME, DATASET_SUBSET, split="validation")
    ds = ds.shuffle(seed=SEED).select(range(n_total))

    tok, model = load_model(hf_name)

    records = []
    for ex in ds:
        question = ex["question"]
        gold_aliases = ex["answer"]["aliases"] + [ex["answer"]["value"]]

        prompt = f"Question: {question}\nAnswer:"
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=tok.pad_token_id,
            )
        gen_text = tok.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

        f1 = best_f1_over_aliases(gen_text, gold_aliases)
        label = int(f1 >= TOKEN_F1_THRESHOLD)

        records.append({
            "question": question, "gold_aliases": gold_aliases,
            "generated": gen_text, "f1": f1, "label": label, "prompt": prompt,
        })

    del model
    torch.cuda.empty_cache()

    with open(cache_path, "wb") as f:
        pickle.dump(records, f)
    print(f"[cached] {cache_path}  (n={len(records)}, factual={sum(r['label'] for r in records)})")
    return records


def train_eval_split(records):
    random.Random(SEED).shuffle(records)
    return records[:N_TRAIN], records[N_TRAIN:N_TRAIN + N_EVAL]


records = generate_and_label(MODEL_KEY, HF_NAME, N_TRAIN + N_EVAL)
train_records, eval_records = train_eval_split(records)
print(f"train={len(train_records)}  eval={len(eval_records)}")

## 3. Hidden state extraction across all layers

Mean-pooled over the generated-answer tokens, cached per model+split.

In [ ]:
@torch.no_grad()
def extract_all_layer_states(cache_key: str, hf_name: str, records, n_layers: int):
    cache_path = os.path.join(CACHE_DIR, f"{cache_key}_hidden_states.pkl")
    if os.path.exists(cache_path):
        print(f"[cache hit] {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    tok, model = load_model(hf_name)

    per_layer_vecs = {layer: [] for layer in range(1, n_layers + 1)}
    labels = []

    for rec in records:
        full_text = rec["prompt"] + " " + rec["generated"]
        inputs = tok(full_text, return_tensors="pt").to(DEVICE)
        prompt_len = tok(rec["prompt"], return_tensors="pt")["input_ids"].shape[1]

        out = model(**inputs, output_hidden_states=True)
        for layer in range(1, n_layers + 1):
            h = out.hidden_states[layer][0]
            answer_span = h[prompt_len:]
            if answer_span.shape[0] == 0:
                answer_span = h[-1:]
            per_layer_vecs[layer].append(answer_span.mean(dim=0).float().cpu().numpy())

        labels.append(rec["label"])

    import numpy as np
    hidden = {layer: np.stack(vecs) for layer, vecs in per_layer_vecs.items()}
    labels = np.array(labels)

    del model
    torch.cuda.empty_cache()

    with open(cache_path, "wb") as f:
        pickle.dump((hidden, labels), f)
    print(f"[cached] {cache_path}")
    return hidden, labels


hidden_train, labels_train = extract_all_layer_states(MODEL_KEY + "_train", HF_NAME, train_records, N_LAYERS)
hidden_eval, labels_eval = extract_all_layer_states(MODEL_KEY + "_eval", HF_NAME, eval_records, N_LAYERS)

## 4. Per-layer linear probes + AUROC sweep (detection side)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler


def robust_auroc(X, y, cv=10, seed=SEED):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=seed)
    aurocs = []
    for train_idx, test_idx in skf.split(X, y):
        scaler = StandardScaler().fit(X[train_idx])
        clf = LogisticRegression(max_iter=2000, C=1.0)
        clf.fit(scaler.transform(X[train_idx]), y[train_idx])
        if len(np.unique(y[test_idx])) < 2:
            continue
        probs = clf.predict_proba(scaler.transform(X[test_idx]))[:, 1]
        aurocs.append(roc_auc_score(y[test_idx], probs))
    return float(np.mean(aurocs)), float(np.std(aurocs))


def sweep_all_layers(hidden, labels, cv=10):
    results = {layer: robust_auroc(X, labels, cv=cv) for layer, X in hidden.items()}
    best_layer = max(results, key=lambda l: results[l][0])
    return results, best_layer


def fit_final_probe(X, y):
    scaler = StandardScaler().fit(X)
    clf = LogisticRegression(max_iter=2000, C=1.0).fit(scaler.transform(X), y)
    return scaler, clf


auroc_results, best_detect_layer = sweep_all_layers(hidden_train, labels_train)

print(f"\n=== Probe AUROC sweep -- {MODEL_KEY} ===")
print(f"{'Layer':>6} {'AUROC':>8} {'Std':>8}")
print("-" * 26)
for layer in sorted(auroc_results.keys()):
    mean, std = auroc_results[layer]
    print(f"{layer:>6} {mean:>8.4f} {std:>8.4f}")

print(f"\nBest detection layer by AUROC: {best_detect_layer} (AUROC={auroc_results[best_detect_layer][0]:.4f})")

import pandas as pd
pd.DataFrame(
    [{"layer": l, "auroc_mean": m, "auroc_std": s} for l, (m, s) in auroc_results.items()]
).sort_values("layer").to_csv(os.path.join(RESULTS_DIR, f"{MODEL_KEY}_probe_auroc.csv"), index=False)

## 5. CAA steering vectors + forward-hook steering mechanism

In [ ]:
def build_caa_vectors(hidden, labels):
    vectors = {}
    for layer, X in hidden.items():
        pos, neg = X[labels == 1], X[labels == 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        diff = pos.mean(axis=0) - neg.mean(axis=0)
        norm = np.linalg.norm(diff)
        unit = diff / norm if norm > 0 else diff
        vectors[layer] = torch.tensor(unit, dtype=torch.float32)
    return vectors


class SteeringHook:
    """Adds alpha * steering_vector to a decoder layer's output during generation."""
    def __init__(self, model, layer_idx, steering_vec, alpha, device, dtype):
        self.vec = steering_vec.to(device=device, dtype=dtype)
        self.alpha = alpha
        target_module = model.model.layers[layer_idx - 1]  # hidden_states index is 1-based
        self.handle = target_module.register_forward_hook(self._hook)

    def _hook(self, module, inputs, output):
        if isinstance(output, tuple):
            hidden_states = output[0] + self.alpha * self.vec
            return (hidden_states,) + output[1:]
        return output + self.alpha * self.vec

    def remove(self):
        self.handle.remove()


def generate_with_optional_steering(model, tok, prompt, device, max_new_tokens,
                                     steer_layer=None, steering_vec=None, alpha=0.0):
    hook = None
    if steer_layer is not None and steering_vec is not None and alpha != 0.0:
        hook = SteeringHook(model, steer_layer, steering_vec, alpha,
                             device=device, dtype=next(model.parameters()).dtype)
    try:
        inputs = tok(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  do_sample=False, pad_token_id=tok.pad_token_id)
        text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    finally:
        if hook is not None:
            hook.remove()
    return text


caa_vectors = build_caa_vectors(hidden_train, labels_train)
print(f"Built CAA vectors for {len(caa_vectors)} layers")

## 6. Held-out accuracy evaluation (with optional steering)

In [ ]:
from tqdm.auto import tqdm

def evaluate_accuracy(model, tok, records_, steer_layer=None, steering_vec=None,
                       alpha=0.0, desc="eval"):
    correct = 0
    for rec in tqdm(records_, desc=desc):
        gen = generate_with_optional_steering(
            model, tok, rec["prompt"], DEVICE, MAX_NEW_TOKENS,
            steer_layer=steer_layer, steering_vec=steering_vec, alpha=alpha,
        )
        f1 = best_f1_over_aliases(gen, rec["gold_aliases"])
        if f1 >= TOKEN_F1_THRESHOLD:
            correct += 1
    return correct / len(records_)

## 7. The layer-swap ablation grid -- THE key experiment

Decouples "which layer detects" from "which layer steers":

1. Use the probe at `detect_layer` to score every eval example, and keep only
   the **boundary-region** cases (probe score in [0.35, 0.65]) -- your
   existing boundary-steering framing, these benefit most from steering.
2. Apply the CAA vector **at `steer_layer`** (built from `steer_layer`'s own
   hidden states) to just those boundary cases.
3. Read off accuracy as a function of `steer_layer`, holding `detect_layer`
   fixed, and vice versa.

If detectability != correctability is real: accuracy should be roughly
**flat** across `detect_layer` and **peaked** across `steer_layer`. If it
instead only looks good when `detect_layer == steer_layer`, the original
finding was a coincidence rather than a general phenomenon.

In [ ]:
def select_boundary_cases(scaler, clf, X_detect, records_, low=0.35, high=0.65):
    probs = clf.predict_proba(scaler.transform(X_detect))[:, 1]
    mask = (probs >= low) & (probs <= high)
    boundary_records = [r for r, m in zip(records_, mask) if m]
    return boundary_records, mask.mean()


def run_layer_swap_grid(model, tok, model_key, hidden_train_, labels_train_,
                         hidden_eval_, eval_records_, caa_vectors_,
                         detect_layers, steer_layers, alpha, baseline_acc=None):
    rows = []
    probes_by_layer = {d: fit_final_probe(hidden_train_[d], labels_train_) for d in detect_layers}

    for d_layer in detect_layers:
        scaler, clf = probes_by_layer[d_layer]
        boundary_records, boundary_frac = select_boundary_cases(
            scaler, clf, hidden_eval_[d_layer], eval_records_
        )
        if len(boundary_records) < 5:
            print(f"[skip] detect_layer={d_layer}: only {len(boundary_records)} boundary cases")
            continue

        for s_layer in steer_layers:
            if s_layer not in caa_vectors_:
                continue
            acc = evaluate_accuracy(
                model, tok, boundary_records, steer_layer=s_layer,
                steering_vec=caa_vectors_[s_layer], alpha=alpha,
                desc=f"detect={d_layer} steer={s_layer}",
            )
            rows.append({
                "model": model_key, "detect_layer": d_layer, "steer_layer": s_layer,
                "alpha": alpha, "n_boundary": len(boundary_records),
                "boundary_frac": boundary_frac, "accuracy": acc,
                "baseline_accuracy": baseline_acc,
                "delta_vs_baseline": (acc - baseline_acc) if baseline_acc is not None else None,
            })
    return pd.DataFrame(rows)


def summarize_swap_pattern(df):
    print("\n=== Accuracy by STEER layer (averaged over detect layers) ===")
    print(df.groupby("steer_layer")["accuracy"].agg(["mean", "std", "count"]))
    print("\n=== Accuracy by DETECT layer (averaged over steer layers) ===")
    print(df.groupby("detect_layer")["accuracy"].agg(["mean", "std", "count"]))

## 8. Run it

Reloads the model for generation, gets baseline accuracy, runs the standard per-layer ablation as a sanity check, then the swap grid.

In [ ]:
mid = N_LAYERS // 2
candidate_layers = sorted(set(
    [max(1, mid - 15), max(1, mid - 10), mid - 5, mid, mid + 5,
     min(N_LAYERS, mid + 10), min(N_LAYERS, N_LAYERS - 2)]
))
candidate_layers = [l for l in candidate_layers if 1 <= l <= N_LAYERS]
print(f"Candidate layers for generation-based eval: {candidate_layers}")

ALPHA = ALPHA_GRID[len(ALPHA_GRID) // 2]  # default to middle of the grid; tune per model
print(f"Using alpha={ALPHA}")

tok, model = load_model(HF_NAME)

In [ ]:
baseline_acc = evaluate_accuracy(model, tok, eval_records, desc=f"{MODEL_KEY} baseline")
print(f"Baseline accuracy: {baseline_acc:.3f}")

In [ ]:
# standard per-layer ablation: same layer used for both detect-boundary-selection and steer
same_layer_rows = []
for layer in candidate_layers:
    if layer not in caa_vectors:
        continue
    acc = evaluate_accuracy(
        model, tok, eval_records, steer_layer=layer,
        steering_vec=caa_vectors[layer], alpha=ALPHA, desc=f"CAA@L{layer}",
    )
    same_layer_rows.append({
        "model": MODEL_KEY, "layer": layer,
        "auroc": auroc_results.get(layer, (None, None))[0],
        "accuracy": acc, "baseline_accuracy": baseline_acc, "delta": acc - baseline_acc,
    })

same_layer_df = pd.DataFrame(same_layer_rows)
same_layer_df.to_csv(os.path.join(RESULTS_DIR, f"{MODEL_KEY}_per_layer_ablation.csv"), index=False)
same_layer_df

In [ ]:
swap_df = run_layer_swap_grid(
    model, tok, MODEL_KEY, hidden_train, labels_train, hidden_eval, eval_records,
    caa_vectors, detect_layers=candidate_layers, steer_layers=candidate_layers,
    alpha=ALPHA, baseline_acc=baseline_acc,
)
swap_df.to_csv(os.path.join(RESULTS_DIR, f"{MODEL_KEY}_layer_swap_grid.csv"), index=False)
summarize_swap_pattern(swap_df)
swap_df

In [ ]:
import json

summary = {
    "model": MODEL_KEY,
    "n_layers": N_LAYERS,
    "best_detect_layer_auroc": best_detect_layer,
    "best_detect_auroc": auroc_results[best_detect_layer][0],
    "baseline_accuracy": baseline_acc,
    "candidate_layers": candidate_layers,
    "alpha_used": ALPHA,
}
with open(os.path.join(RESULTS_DIR, f"{MODEL_KEY}_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"DONE for {MODEL_KEY}. Now change MODEL_KEY in the config cell above to the next")
print("model, re-run the whole notebook (cached stages will be skipped), and repeat")
print("for all three models before running the cross-model comparison below.")
summary

## 9. Cross-model comparison

Only run this once `run_experiment` results exist for **all three** models
(re-run sections 1-8 above with `MODEL_KEY` set to each of the three). Pools
the layer-swap grids and checks whether the flat-by-detect / peaked-by-steer
pattern generalizes.

In [ ]:
def load_all_swap_grids():
    dfs = []
    for model_key in MODEL_CONFIGS:
        path = os.path.join(RESULTS_DIR, f"{model_key}_layer_swap_grid.csv")
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
        else:
            print(f"[missing] {path} -- run this notebook with MODEL_KEY='{model_key}' first")
    if not dfs:
        raise FileNotFoundError("No layer-swap grid results found in ./results/")
    return pd.concat(dfs, ignore_index=True)


def normalize_layer_depth(df):
    n_layers_map = {k: v["n_layers"] for k, v in MODEL_CONFIGS.items()}
    df["n_layers_total"] = df["model"].map(n_layers_map)
    df["detect_depth_pct"] = df["detect_layer"] / df["n_layers_total"]
    df["steer_depth_pct"] = df["steer_layer"] / df["n_layers_total"]
    return df


all_df = load_all_swap_grids()
all_df = normalize_layer_depth(all_df)
all_df.to_csv(os.path.join(RESULTS_DIR, "all_models_layer_swap_grid.csv"), index=False)

print("\n=== Cross-model: accuracy by STEER depth %, averaged over detect layers ===")
all_df["steer_depth_bucket"] = (all_df["steer_depth_pct"] * 10).round() / 10
print(all_df.groupby(["model", "steer_depth_bucket"])["accuracy"].mean().unstack("model"))

print("\n=== Cross-model: accuracy by DETECT depth %, averaged over steer layers ===")
all_df["detect_depth_bucket"] = (all_df["detect_depth_pct"] * 10).round() / 10
print(all_df.groupby(["model", "detect_depth_bucket"])["accuracy"].mean().unstack("model"))

print(
    "\nInterpretation guide:\n"
    "  - STEER table: a consistent peak around the same depth % across all three\n"
    "    models = cross-model evidence for a correction-layer sweet spot.\n"
    "  - DETECT table: roughly flat across all three models = detection-layer\n"
    "    choice doesn't matter for correction quality, i.e. detectability\n"
    "    genuinely != correctability, not just a Llama/TriviaQA coincidence.\n"
)
all_df